# Azure Machine Learning — Deploying a Model to a Managed Online Endpoint

Azure Machine Learning (Azure ML) is Microsoft's managed ML platform. It organises models, datasets, compute, and deployments inside a **workspace** — a central resource group for all ML activities.

This notebook covers the full deployment flow using the `azure-ai-ml` SDK. All calls that require Azure credentials are wrapped in `try/except` so the notebook runs cleanly in any environment.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain what an Azure ML workspace, online endpoint, and deployment are
2. Write and understand the scoring script pattern (`init()` and `run()`)
3. Register a model in the Azure ML model registry
4. Create an online endpoint and deploy a model to it
5. Invoke the endpoint and interpret the response

## 1. Azure ML Concepts

**Workspace** — the top-level Azure ML resource. Contains all assets: models, datasets, compute clusters, endpoints.

**Online Endpoint** — a stable HTTPS URL. Traffic arrives here. Think of it as the address for your service.

**Deployment** — the actual running infrastructure behind an endpoint. One endpoint can have multiple deployments with traffic splits (useful for A/B testing).

```
Workspace
  └── Online Endpoint: https://iris-endpoint.eastus.inference.ml.azure.com
        ├── Deployment: blue  (90% traffic) -- current production model
        └── Deployment: green (10% traffic) -- candidate new model
```

**DefaultAzureCredential** tries multiple auth methods in order:
1. Environment variables (`AZURE_CLIENT_ID`, `AZURE_TENANT_ID`, `AZURE_CLIENT_SECRET`)
2. Managed Identity (when running inside an Azure VM or container)
3. Azure CLI token (`az login`)
4. Visual Studio Code extension token
5. Interactive browser login

## 2. Install Dependencies

In [1]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'azure-ai-ml', 'azure-identity', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('azure-ai-ml and azure-identity installed successfully')
else:
    print(result.stdout[-500:] if result.stdout else result.stderr[-500:])

azure-ai-ml and azure-identity installed successfully


## 3. Train and Save the Model

In [2]:
import joblib
import pathlib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_s, y_train)

print(f"Test accuracy: {clf.score(scaler.transform(X_test), y_test):.2%}")

# Save to a dedicated model directory (Azure ML convention)
model_dir = pathlib.Path('/tmp/azure_model')
model_dir.mkdir(exist_ok=True)
joblib.dump(clf, model_dir / 'model.joblib')
joblib.dump(scaler, model_dir / 'scaler.joblib')
print(f"Model artifacts saved to {model_dir}")

<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


Test accuracy: 100.00%
Model artifacts saved to /tmp/azure_model


## 4. The Scoring Script

Azure ML requires an inference script with two functions:
- `init()` — called once when the container starts; load the model here
- `run(data)` — called for every request; receives JSON, returns JSON

This script gets packaged with the deployment and runs inside the managed container.

In [3]:
scoring_script = '''\
import os
import json
import numpy as np
import joblib

# Global variables - loaded once in init()
model = None
scaler = None
CLASSES = ["setosa", "versicolor", "virginica"]

def init():
    """Called once when the container starts. Load model artifacts here."""
    global model, scaler
    # AZUREML_MODEL_DIR is set by the runtime to the model directory
    model_dir = os.environ.get("AZUREML_MODEL_DIR", "/tmp/azure_model")
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    scaler = joblib.load(os.path.join(model_dir, "scaler.joblib"))
    print("Model loaded from:", model_dir)

def run(raw_data):
    """Called for every request. Returns prediction as JSON string."""
    try:
        data = json.loads(raw_data)
        # Expect: {"data": [[5.1, 3.5, 1.4, 0.2], ...]}
        features = np.array(data["data"])
        features_scaled = scaler.transform(features)
        predictions = model.predict(features_scaled).tolist()
        probabilities = model.predict_proba(features_scaled).max(axis=1).tolist()
        labels = [CLASSES[p] for p in predictions]
        return json.dumps({
            "predictions": labels,
            "confidence": [round(p, 3) for p in probabilities]
        })
    except Exception as e:
        return json.dumps({"error": str(e)})
'''

score_path = pathlib.Path('/tmp/azure_model/score.py')
score_path.write_text(scoring_script)
print(f"Scoring script written to: {score_path}")
print("\nAzure ML requires two functions:")
print("  init() -- called once at container startup, load model here")
print("  run()  -- called for each prediction request")

Scoring script written to: /tmp/azure_model/score.py

Azure ML requires two functions:
  init() -- called once at container startup, load model here
  run()  -- called for each prediction request


In [4]:
# Test the scoring script locally before deploying
import importlib.util, os, json

os.environ['AZUREML_MODEL_DIR'] = '/tmp/azure_model'
spec = importlib.util.spec_from_file_location('score', '/tmp/azure_model/score.py')
score_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score_module)

score_module.init()

test_input = json.dumps({'data': [[5.1, 3.5, 1.4, 0.2], [6.7, 3.1, 4.7, 1.5]]})
result = score_module.run(test_input)
print("Local scoring test result:")
print(json.dumps(json.loads(result), indent=2))

Model loaded from: /tmp/azure_model
Local scoring test result:
{
  "predictions": [
    "setosa",
    "versicolor"
  ],
  "confidence": [
    1.0,
    1.0
  ]
}


## 5. Connect to an Azure ML Workspace

In [5]:
try:
    from azure.ai.ml import MLClient
    from azure.identity import DefaultAzureCredential

    SUBSCRIPTION_ID = 'your-subscription-id'
    RESOURCE_GROUP = 'my-ml-resource-group'
    WORKSPACE_NAME = 'my-ml-workspace'

    credential = DefaultAzureCredential()
    ml_client = MLClient(
        credential=credential,
        subscription_id=SUBSCRIPTION_ID,
        resource_group_name=RESOURCE_GROUP,
        workspace_name=WORKSPACE_NAME,
    )
    print(f"Connected to workspace: {ml_client.workspace_name}")

except Exception as e:
    ml_client = None
    print(f"[Auth error - expected in local environment] {type(e).__name__}")
    print()
    print("DefaultAzureCredential auth order:")
    methods = [
        'Environment variables (AZURE_CLIENT_ID, AZURE_TENANT_ID, AZURE_CLIENT_SECRET)',
        'Managed Identity (works inside Azure VMs, App Service, AKS)',
        'Azure CLI token (run: az login)',
        'Visual Studio Code extension token',
        'Interactive browser login',
    ]
    for i, method in enumerate(methods, 1):
        print(f"  {i}. {method}")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Connected to workspace: my-ml-workspace


## 6. Register the Model

In [6]:
try:
    from azure.ai.ml.entities import Model
    from azure.ai.ml.constants import AssetTypes

    model_asset = Model(
        path='/tmp/azure_model',
        type=AssetTypes.CUSTOM_MODEL,
        name='iris-classifier',
        version='1',
        description='Random Forest iris classifier with StandardScaler',
    )

    if ml_client:
        registered_model = ml_client.models.create_or_update(model_asset)
        print(f"Model registered: {registered_model.name} v{registered_model.version}")
    else:
        print("[No client] Would call: ml_client.models.create_or_update(model_asset)")
        print("Model config:")
        print(f"  name    : {model_asset.name}")
        print(f"  version : {model_asset.version}")
        print(f"  path    : {model_asset.path}")
        print(f"  type    : {model_asset.type}")

except Exception as e:
    print(f"[Error] {type(e).__name__}: {e}")

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/py

[Error] ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guideli

## 7. Create an Online Endpoint

In [7]:
ENDPOINT_NAME = 'iris-endpoint-aiat125'

try:
    from azure.ai.ml.entities import ManagedOnlineEndpoint

    endpoint = ManagedOnlineEndpoint(
        name=ENDPOINT_NAME,
        description='Iris species classifier endpoint',
        auth_mode='key',   # 'key' = static API key auth
        tags={'course': 'AIAT125', 'model': 'iris-rf'},
    )

    if ml_client:
        poller = ml_client.online_endpoints.begin_create_or_update(endpoint)
        result = poller.result()  # blocks until done (~2 min)
        print(f"Endpoint created: {result.scoring_uri}")
    else:
        print("[No client] Would call: ml_client.online_endpoints.begin_create_or_update(endpoint)")
        print(f"  name      : {ENDPOINT_NAME}")
        print(f"  auth_mode : key (requests must include the endpoint API key)")
        print(f"  scoring URI would be: https://{ENDPOINT_NAME}.eastus.inference.ml.azure.com/score")

except Exception as e:
    print(f"[Error] {type(e).__name__}: {e}")

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/py

[Error] ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guideli

## 8. Create a Deployment and Route Traffic

In [8]:
try:
    from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

    deployment = ManagedOnlineDeployment(
        name='blue',
        endpoint_name=ENDPOINT_NAME,
        model='azureml:iris-classifier:1',      # references the registered model
        code_configuration=CodeConfiguration(
            code='/tmp/azure_model',
            scoring_script='score.py',
        ),
        environment='azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu:33',
        instance_type='Standard_DS2_v2',  # 2 vCPU, 7 GB RAM
        instance_count=1,
    )

    if ml_client:
        poller = ml_client.online_deployments.begin_create_or_update(deployment)
        poller.result()
        # Route 100% of traffic to the blue deployment
        endpoint_obj = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint_obj.traffic = {'blue': 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint_obj).result()
        print("Deployment created and 100% traffic routed to 'blue'.")
    else:
        print("[No client] Would call: ml_client.online_deployments.begin_create_or_update(deployment)")
        print("Deployment config:")
        print(f"  name           : blue")
        print(f"  instance_type  : Standard_DS2_v2")
        print(f"  instance_count : 1")
        print(f"  scoring_script : score.py")
        print()
        print("A/B test traffic split example:")
        print("  endpoint.traffic = {'blue': 90, 'green': 10}")
        print("  ml_client.online_endpoints.begin_create_or_update(endpoint)")

except Exception as e:
    print(f"[Error] {type(e).__name__}: {e}")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint iris-endpoint-aiat125 exists
DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	A

[Error] ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guideli

## 9. Invoke the Endpoint

In [9]:
import json

test_payload = json.dumps({'data': [[5.1, 3.5, 1.4, 0.2], [6.7, 3.1, 4.7, 1.5]]})

try:
    if ml_client:
        response = ml_client.online_endpoints.invoke(
            endpoint_name=ENDPOINT_NAME,
            deployment_name='blue',
            body=test_payload,
        )
        print(f"Response: {response}")
    else:
        print("[No client] Would call: ml_client.online_endpoints.invoke(...)")
        print(f"Input: {test_payload}")
        print()
        # Simulate using local model
        data = json.loads(test_payload)
        features = np.array(data['data'])
        features_s = scaler.transform(features)
        preds = clf.predict(features_s)
        probs = clf.predict_proba(features_s).max(axis=1)
        simulated = {
            'predictions': [iris.target_names[p] for p in preds],
            'confidence': [round(float(p), 3) for p in probs]
        }
        print(f"Simulated response: {json.dumps(simulated, indent=2)}")

except Exception as e:
    print(f"[Error] {type(e).__name__}: {e}")

[Error] TypeError: expected str, bytes or os.PathLike object, not NoneType


## 10. Clean Up

In [10]:
try:
    if ml_client:
        ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
        print(f"Endpoint '{ENDPOINT_NAME}' deleted.")
    else:
        print("[No client] Would call: ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME)")
        print()
        print("Cleanup order:")
        print("  1. Delete deployment  -- stops billing for compute instances")
        print("  2. Delete endpoint    -- removes the DNS/URL")
        print("  3. Optionally delete the registered model version")

except Exception as e:
    print(f"[Error] {type(e).__name__}: {e}")

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/py

[Error] ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guideli

## Summary

The Azure ML online endpoint deployment flow:
1. **Train & save** the model locally
2. **Write a scoring script** with `init()` and `run()` — Azure ML calls these functions
3. **Register the model** in the Azure ML model registry (`ml_client.models.create_or_update`)
4. **Create an online endpoint** — the stable HTTPS URL
5. **Create a deployment** — specifies instance type, model version, scoring script
6. **Route traffic** — assign percentages to deployments (enables A/B testing)
7. **Delete** when done to stop billing

Key Azure ML vocabulary:
- **Endpoint** = the URL (stable, does not change when you update the model)
- **Deployment** = the running infrastructure behind the endpoint (changes when you update)

## Self-Check

1. **What is the scoring script in Azure ML?**
   *(What two functions must it define, and when is each called?)*

2. **What does `DefaultAzureCredential` try in order to authenticate?**
   *(List at least three methods it attempts, in order.)*

3. **How is an Azure ML 'deployment' different from an 'endpoint'?**
   *(Which one changes when you push a new model version? Which stays the same?)*